# EgoDex Scenario Search on Daft

This notebook reproduces the workflow behind the EgoDex scenario-search blog post with the cleaned `egodex` package. It reads raw EgoDex HDF5 episodes directly, computes hand-pose feature tracks, embeds sampled video frames with SigLIP-2 through Daft, then runs pose-only, text-only, and combined pose plus semantic queries.

Point `DATA_ROOT` at the extracted ~16 GB EgoDex download and use `EPISODE_LIMIT` to bound how many episodes each run processes.

## 0. Setup

From the repository root, install the editable package and notebook kernel once:

```bash
uv sync --extra egodex --extra notebook
```

Select the repo `.venv` as the Jupyter kernel. The notebook resolves paths from the repo root automatically, so the kernel cwd can be anywhere under the checkout.

In [ ]:
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "datasets" / "egodex").exists():
            return candidate
    raise RuntimeError("Could not find the daft-examples repository root.")


def resolve_data_root(repo_root: Path) -> Path:
    candidates = (
        repo_root / "datasets" / "egodex" / ".data",
        repo_root / ".data",
        Path(".data"),
    )
    for path in candidates:
        if path.exists() and any(path.glob("**/*.hdf5")):
            return path.resolve()
    raise FileNotFoundError(
        "EgoDex HDF5 episodes not found. Extract the dataset to "
        f"{repo_root / 'datasets' / 'egodex' / '.data'} before running this notebook."
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
DATA_ROOT = resolve_data_root(REPO_ROOT)
FEATURES_DIR = REPO_ROOT / "datasets" / "egodex" / ".features"
EMBEDDINGS_DIR = REPO_ROOT / "datasets" / "egodex" / ".embeddings"

print(f"repo root: {REPO_ROOT}")
print(f"data root: {DATA_ROOT}")

In [ ]:
import os

# Quiet Daft and HuggingFace progress bars before any pipeline work.
os.environ.setdefault("DAFT_PROGRESS_BAR", "0")
os.environ.setdefault("DAFT_STA", "error")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")

import daft
from egodex import (
    FPS,
    SCENARIOS,
    EgoDexPipeline,
    calibrate,
    overlay,
    pose_search_many,
    query,
)

EPISODE_LIMIT = 10
PREVIEW_SECONDS = 5.0
TOP_K = 3

# See how EgoDexPipeline is implemented in pipeline.py
pipeline = EgoDexPipeline(
    str(DATA_ROOT),
    features_dir=str(FEATURES_DIR),
    embeddings_dir=str(EMBEDDINGS_DIR),
)

print(f"Daft {daft.__version__}")
print(f"episode limit: {EPISODE_LIMIT}")

## 1. Direct HDF5 Episode Read

`raw()` discovers every `<task>/<episode>.hdf5` file under `DATA_ROOT` and attaches the sibling MP4 as a lazy video file. `EPISODE_LIMIT` trims the episode table before any metadata or trajectory tensors are read.

In [ ]:
episodes = pipeline.raw()
if EPISODE_LIMIT is not None:
    episodes = episodes.limit(int(EPISODE_LIMIT))

episodes.select("task", "episode_id", "video").show(10)

## 2. Pose Feature Branch

The pose branch stores **one row per frame**. `trajectory()` reads only the transform tensors needed for pose features, and the feature stage runs in two steps:

1. `frame_features()` — an episode-level UDF computes the instantaneous hand geometry (closure, grip distances, wrist position, ...) and explodes it into per-frame rows keyed by `(task, episode_id, frame_index)`.
2. `temporal_features()` — every action rate (grasping, lifting, reaching, twisting, ...) is a **Daft window expression** over `Window().partition_by("task", "episode_id").order_by("frame_index")`: next-frame diffs via `lead(1)`, point speeds via `euclidean_distance`, and smoothing via a centered `rows_between(-2, 2)` mean. The one custom UDF, `forearm_roll`, is fed by the same window — this frame's wrist rotation plus the next frame's via `lead(1)`.

Because the rates are expressions, the temporal math stays in the query plan — no collect, and the same code applies to any per-frame table with these columns (e.g. LeRobot-style parquet). Query-time scenarios group the frames back into per-episode tracks and stitch matching frames into segments.

`calculate_features()` runs both steps at once.

In [ ]:
# Read the raw trajectories, then build features in two stages:
# per-frame spatial geometry, plus action rates as Daft window expressions.
trajectories = pipeline.trajectory(episodes)
spatial = pipeline.frame_features(trajectories)
features = pipeline.temporal_features(spatial)  # see temporal.py for the window expressions

# One row per frame; rates like wrist_vert_vel/roll come from lead(1) windows.
features.select("task", "episode_id", "frame_index", "closure_L", "wrist_vert_vel_L", "roll_L").show(10)
features.write_parquet(str(FEATURES_DIR))
print(f"wrote features to {FEATURES_DIR}")

## 3. Scenario Vocabulary

The blog separates physical states from actions. States read a frame's hand shape; actions read motion between frames. Thresholds are calibrated from continuous feature tracks and reused across queries.

In [ ]:
SCENARIO_CATALOG = [
    {"scenario": "openness", "kind": "state", "signal": "closure band"},
    {"scenario": "writing_grip", "kind": "state", "signal": "tripod grip geometry"},
    {"scenario": "hammer_grip", "kind": "state", "signal": "power grip geometry"},
    {"scenario": "twisting", "kind": "action", "signal": "forearm roll rate"},
    {"scenario": "reaching", "kind": "action", "signal": "arm extension rate"},
    {"scenario": "lifting", "kind": "action", "signal": "wrist vertical velocity"},
    {"scenario": "grasping", "kind": "action", "signal": "curl closing rate"},
    {"scenario": "in_hand", "kind": "action", "signal": "still wrist plus finger articulation"},
]

assert set(item["scenario"] for item in SCENARIO_CATALOG) <= set(SCENARIOS)
daft.from_pylist(SCENARIO_CATALOG).show()

In [ ]:
thresholds = calibrate(features)
daft.from_pylist([{key: round(value, 4) for key, value in thresholds.items()}])

## 4. Pose-Only Search

Pose-only queries rank episodes by matching frame count inside Daft. Each scenario is a lazy plan: the match UDF scans episode tracks, then Daft sorts and limits the top hits.

In [ ]:
POSE_QUERIES = [
    ("open hands", {"pose": "openness", "open_lo": 0.65, "open_hi": 1.0}),
    ("closed hands", {"pose": "openness", "open_lo": 0.0, "open_hi": 0.35}),
    ("writing grip", {"pose": "writing_grip"}),
    ("hammer grip", {"pose": "hammer_grip"}),
    ("twisting", {"pose": "twisting"}),
    ("reaching", {"pose": "reaching"}),
    ("lifting", {"pose": "lifting"}),
    ("grasping", {"pose": "grasping"}),
    ("in-hand manipulation", {"pose": "in_hand"}),
]

pose_hits = pose_search_many(features, POSE_QUERIES, k=TOP_K, thresholds=thresholds)
pose_hits.select("query", "task", "episode_id", "score", "segments").show()

## 5. Video Frame Sampling

The semantic branch samples video frames at about 1 fps. This preview decodes only the first few seconds so the cell stays bounded.

In [ ]:
preview_frames = pipeline.camera_frames(
    episodes,
    end_time=PREVIEW_SECONDS,
    sample_interval_seconds=1.0,
)
preview_frames.select("task", "episode_id", "video_frames").explode("video_frames").select(
    "task",
    "episode_id",
    daft.col("video_frames")["frame_index"].alias("frame_index"),
    daft.col("video_frames")["frame_time"].alias("frame_time"),
).show()

## 6. SigLIP Embeddings

Embed sampled frames with SigLIP-2 through a batch UDF. The first run downloads and loads model weights.

In [ ]:
embedding_frames = pipeline.camera_frames(
    episodes,
    sample_interval_seconds=pipeline.sample_interval_seconds,
)
embeddings = pipeline.embed_frames(embedding_frames)
embeddings.select("task", "episode_id", "frame_index", "timestamp", "clip_emb").show(3)
embeddings.write_parquet(str(EMBEDDINGS_DIR))
print(f"wrote embeddings to {EMBEDDINGS_DIR}")

## 7. Text and Combined Search

Text-only search ranks sampled frame embeddings by SigLIP similarity and returns a short window around the best frame. Combined search first applies the pose mask, then ranks matching sampled frames by text similarity.

In [ ]:
TEXT_QUERIES = [
    ("text: chopsticks", {"text": "chopsticks"}),
    ("text: stapler", {"text": "stapler"}),
    ("hammer grip + stapler", {"pose": "hammer_grip", "text": "stapler"}),
    ("grasping + shirt", {"pose": "grasping", "text": "shirt"}),
    ("reaching + marbles", {"pose": "reaching", "text": "marbles"}),
]

text_rows = []
for label, params in TEXT_QUERIES:
    for hit in query(features, clip=embeddings, k=TOP_K, thresholds=thresholds, **params):
        text_rows.append(
            {
                "query": label,
                **{key: value for key, value in hit.items() if key != "segments"},
                "segments": [[int(start), int(end)] for start, end in hit["segments"]],
            }
        )

text_hits = daft.from_pylist(text_rows)
text_hits.select("query", "task", "episode_id", "score", "segments").show()

## 8. Segment Inspection

Use the returned segments to inspect exact frames. This mirrors the blog's dashboard behavior in notebook form: pick a hit, list its segments, then overlay the HDF5 skeleton on frames from the first segment.

In [ ]:
def _segment_bounds(segment):
    if isinstance(segment, dict):
        return int(segment["_0"]), int(segment["_1"])
    return int(segment[0]), int(segment[1])


def first_hit(hits_df):
    if "query" not in hits_df.column_names:
        return None, None
    row = hits_df.sort("score", desc=True).limit(1).to_pydict()
    if not row["query"] or row["query"][0] is None:
        return None, None
    return row["query"][0], {
        "task": row["task"][0],
        "episode_id": int(row["episode_id"][0]),
        "score": float(row["score"][0]),
        "segments": [_segment_bounds(segment) for segment in row["segments"][0]],
    }


def first_available_hit(*hit_tables):
    for hits_df in hit_tables:
        label, hit = first_hit(hits_df)
        if hit is not None:
            return label, hit
    return None, None


selected_label, selected_hit = first_available_hit(text_hits, pose_hits)
selected_label, selected_hit

In [ ]:
if selected_hit is None:
    segment_table = []
else:
    segment_table = [
        {
            "segment": index,
            "start_frame": start,
            "end_frame": end,
            "start_seconds": round(start / FPS, 3),
            "end_seconds": round(end / FPS, 3),
            "duration_seconds": round((end - start + 1) / FPS, 3),
        }
        for index, (start, end) in enumerate(selected_hit["segments"], start=1)
    ]
segment_table

In [ ]:
from IPython.display import display

if selected_hit is not None and selected_hit["segments"]:
    start, end = selected_hit["segments"][0]
    midpoint = (start + end) // 2
    frame_indices = sorted({start, midpoint, end})
    print(f"{selected_label}: {selected_hit['task']}/{selected_hit['episode_id']} segment {start}-{end}")
    for frame_index in frame_indices:
        print(f"frame {frame_index}")
        display(
            overlay(
                DATA_ROOT,
                selected_hit["task"],
                selected_hit["episode_id"],
                frame_index,
                radius=3,
            )
        )
else:
    print("No segment available to visualize.")

## 9. Scaling Up

Increase `EPISODE_LIMIT` for broader search, or set it to `None` to process every episode under `DATA_ROOT`. Feature and embedding parquets land in `FEATURES_DIR` and `EMBEDDINGS_DIR` so you can reload them in a later session with `daft.read_parquet(...)` instead of recomputing.